# Part D: Risk Measurement & Stress Analysis

**Note:** All interpretations are based on outputs generated as of **April 14, 2026**.  
Market data is fetched programmatically via `yfinance` for a rolling **6-month window** (Oct 14, 2025 – Apr 13, 2026, 122 trading days).  
Stocks: **HDFCBANK.NS** (Liquid) | **NESTLEIND.NS** (Illiquid)


In [ ]:
!pip install arch --quiet

## Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox
from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
%matplotlib inline

LIQUID   = 'HDFCBANK.NS'
ILLIQUID = 'NESTLEIND.NS'

END_DATE    = pd.Timestamp.today().normalize()
START_DATE  = END_DATE - pd.DateOffset(months=6)
BUFFER_DATE = START_DATE - pd.DateOffset(days=45)   # warm-up for rolling window

ROLLING_WINDOW    = 20      # 20-day rolling volatility window
HIGH_VOL_QUANTILE = 0.75    # top 25% rolling-vol days = High-Vol regime
Z_95, Z_99        = 1.6449, 2.3263
INVESTMENT        = 100_000

print(f'Analysis window : {START_DATE.date()} → {END_DATE.date()}')

## Data Download

We download 45 extra calendar days before `START_DATE` so the 20-day rolling window
has warm-up observations. Log returns are computed on this extended series, then trimmed
to the 6-month analysis window — ensuring every day has a valid rolling volatility value.

In [ ]:
raw = yf.download(
    [LIQUID, ILLIQUID],
    start=BUFFER_DATE, end=END_DATE,
    auto_adjust=True, progress=False
)['Close'].rename(columns={LIQUID: 'HDFCBANK', ILLIQUID: 'NESTLEIND'})

raw = raw.dropna()
raw = raw[~raw.index.duplicated()].sort_index()

# Log returns on extended series
log_ret_ext = np.log(raw / raw.shift(1)).dropna()

# Trim to 6-month analysis window
ret = log_ret_ext[log_ret_ext.index >= START_DATE].copy()

print(f'Extended series : {log_ret_ext.index[0].date()} → {log_ret_ext.index[-1].date()}')
print(f'Analysis window : {ret.index[0].date()} → {ret.index[-1].date()}  ({len(ret)} trading days)')
display(ret.head())

## Section A: Parametric VaR — Model Building Approach

The model building approach assumes returns are normally distributed and estimates VaR analytically:

$$\text{VaR}_{\alpha} = -(\mu - z_{\alpha} \cdot \sigma)$$

- $\mu$ and $\sigma$ are estimated from the 6-month log-return sample  
- $z_{95\%} = 1.6449$, $z_{99\%} = 2.3263$ (one-tailed)  
- VaR is a **positive number** representing the maximum expected 1-day loss at the given confidence level

In [ ]:
def parametric_var(series, z):
    mu, sigma = series.mean(), series.std(ddof=1)
    return -(mu - z * sigma), mu, sigma

rows = []
for label, col in [('HDFCBANK (Liquid)', 'HDFCBANK'), ('NESTLEIND (Illiquid)', 'NESTLEIND')]:
    for conf, z in [('95%', Z_95), ('99%', Z_99)]:
        var, mu, sigma = parametric_var(ret[col], z)
        rows.append({
            'Stock'              : label,
            'Confidence'         : conf,
            'Mean Return (%)'    : round(mu * 100, 4),
            'Daily σ (%)'        : round(sigma * 100, 4),
            'VaR (%)'            : round(var * 100, 4),
            'VaR (Rs. on Rs.1L)' : round(var * INVESTMENT, 2)
        })

df_var = pd.DataFrame(rows)
print('Full-Period Parametric VaR (Model Building Approach)')
display(df_var)

## Section B: Regime Classification — Normal vs High-Volatility

The assignment requires comparing VaR across normal and high-volatility periods,
where the **top 25% of days by 20-day rolling volatility** are classified as High-Vol.

A single full-period VaR blends calm and turbulent days — computing VaR separately
per regime reveals how much risk is understated during stress.

In [ ]:
def classify_regimes(col):
    roll_vol = log_ret_ext[col].rolling(ROLLING_WINDOW).std()
    roll_vol = roll_vol[roll_vol.index >= START_DATE].dropna()
    threshold   = roll_vol.quantile(HIGH_VOL_QUANTILE)
    high_mask   = roll_vol > threshold
    r           = ret[col].loc[roll_vol.index]
    return r[~high_mask], r[high_mask], threshold

for label, col in [('HDFCBANK (Liquid)', 'HDFCBANK'), ('NESTLEIND (Illiquid)', 'NESTLEIND')]:
    rn, rh, thr = classify_regimes(col)
    print(f'{label}')
    print(f'  75th pct threshold : {thr*100:.4f}%')
    print(f'  Normal days        : {len(rn)}')
    print(f'  High-Vol days      : {len(rh)}')
    print()

In [ ]:
regime_rows = []
for label, col in [('HDFCBANK (Liquid)', 'HDFCBANK'), ('NESTLEIND (Illiquid)', 'NESTLEIND')]:
    ret_n, ret_h, _ = classify_regimes(col)
    for regime, r in [('Normal', ret_n), ('High-Vol', ret_h)]:
        for conf, z in [('95%', Z_95), ('99%', Z_99)]:
            var, mu, sigma = parametric_var(r, z)
            regime_rows.append({
                'Stock'        : label,
                'Regime'       : regime,
                'n Days'       : len(r),
                'Confidence'   : conf,
                'σ (%)'        : round(sigma * 100, 4),
                'VaR (%)'      : round(var * 100, 4),
                'VaR (Rs. 1L)' : round(var * INVESTMENT, 2)
            })

df_regime = pd.DataFrame(regime_rows)
print('Regime VaR: Normal vs High-Volatility')
display(df_regime)

## Section C: Normality Test — Jarque-Bera

Parametric VaR assumes normally distributed returns. The Jarque-Bera test checks this assumption independently for each stock using the 6-month data:

$$H_0: \text{Returns are normal} \qquad H_1: \text{Skewness or fat tails present}$$

- $p < 0.05$ → reject normality → basic parametric VaR understates true tail risk
- Rejection motivates the optional GARCH and Monte Carlo extensions

In [ ]:
norm_rows = []
for label, col in [('HDFCBANK (Liquid)', 'HDFCBANK'), ('NESTLEIND (Illiquid)', 'NESTLEIND')]:
    r = ret[col].dropna()
    jb_stat, jb_p = stats.jarque_bera(r)
    norm_rows.append({
        'Stock'          : label,
        'Skewness'       : round(r.skew(), 4),
        'Excess Kurtosis': round(r.kurtosis(), 4),
        'JB Stat'        : round(jb_stat, 4),
        'p-value'        : round(jb_p, 6),
        'Normal?'        : 'Yes' if jb_p >= 0.05 else 'No — Fat Tails Detected'
    })

df_norm = pd.DataFrame(norm_rows)
print('Normality Test (Jarque-Bera)')
display(df_norm)

## Visualizations

In [ ]:
# Return distributions with Normal PDF overlay and VaR lines
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
COLORS = {'HDFCBANK': 'steelblue', 'NESTLEIND': 'darkorange'}
LABELS = {'HDFCBANK': 'HDFCBANK (Liquid)', 'NESTLEIND': 'NESTLEIND (Illiquid)'}

for idx, col in enumerate(['HDFCBANK', 'NESTLEIND']):
    ax, r = axes[idx], ret[col]
    ax.hist(r, bins=50, color=COLORS[col], alpha=0.55, density=True, edgecolor='none', label='Empirical')
    x = np.linspace(r.min(), r.max(), 300)
    ax.plot(x, stats.norm.pdf(x, r.mean(), r.std()), 'k--', lw=1.8, label='Normal PDF')
    for conf, z, color in [('95%', Z_95, 'green'), ('99%', Z_99, 'crimson')]:
        var = -(r.mean() - z * r.std())
        ax.axvline(-var, color=color, lw=1.8, ls=':', label=f'VaR {conf} = {var*100:.2f}%')
    ax.set_title(f'{LABELS[col]}\nReturn Distribution + VaR Thresholds', fontsize=11)
    ax.set_xlabel('Daily Log Return'); ax.set_ylabel('Density'); ax.legend(fontsize=8)

plt.suptitle('Return Distributions with Normal PDF and VaR Thresholds', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# Regime VaR bar chart
fig, ax = plt.subplots(figsize=(14, 6))
combos     = [('Normal','95%',Z_95), ('Normal','99%',Z_99),
              ('High-Vol','95%',Z_95), ('High-Vol','99%',Z_99)]
bar_colors = ['#2ecc71', '#27ae60', '#e67e22', '#c0392b']
x_pos      = np.arange(2)
stock_keys = [('HDFCBANK (Liquid)', 'HDFCBANK'), ('NESTLEIND (Illiquid)', 'NESTLEIND')]

for (regime, conf, z), offset, bc in zip(combos, [-1.5, -0.5, 0.5, 1.5], bar_colors):
    vals = []
    for _, col in stock_keys:
        rn, rh, _ = classify_regimes(col)
        r = rn if regime == 'Normal' else rh
        v, _, _ = parametric_var(r, z)
        vals.append(v * 100)
    bars = ax.bar(x_pos + offset * 0.18, vals, 0.16, color=bc, alpha=0.85, label=f'{regime} {conf}')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.2f}%', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x_pos)
ax.set_xticklabels([k[0] for k in stock_keys], fontsize=11)
ax.set_ylabel('1-Day VaR (%)'); ax.set_title('1-Day VaR by Regime and Confidence Level')
ax.legend(ncol=4, fontsize=9); plt.tight_layout(); plt.show()

In [ ]:
# Rolling volatility timeline with high-vol regime shading
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for idx, (col, label) in enumerate([('HDFCBANK', 'HDFCBANK (Liquid)'),
                                     ('NESTLEIND', 'NESTLEIND (Illiquid)')]):
    ax        = axes[idx]
    roll_vol  = log_ret_ext[col].rolling(ROLLING_WINDOW).std()
    roll_vol  = roll_vol[roll_vol.index >= START_DATE].dropna()
    threshold = roll_vol.quantile(HIGH_VOL_QUANTILE)
    color     = 'steelblue' if col == 'HDFCBANK' else 'darkorange'

    ax.plot(roll_vol.index, roll_vol * 100, lw=1.5, color=color, label='20-day Rolling σ')
    ax.axhline(threshold * 100, color='crimson', ls='--', lw=1.5,
               label=f'75th pct = {threshold*100:.3f}%')
    for d in roll_vol[roll_vol > threshold].index:
        ax.axvspan(d, d, alpha=0.15, color='red')
    ax.set_title(f'{label} — Rolling Volatility & Regime Classification', fontsize=11)
    ax.set_ylabel('Daily σ (%)'); ax.legend(fontsize=9)

axes[-1].set_xlabel('Date')
plt.suptitle('20-Day Rolling Volatility with High-Vol Regime Shading', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

---
## (Optional) Section D: GARCH(1,1) Parametric VaR — Time-Varying Volatility

### Theory

Basic parametric VaR assumes constant volatility (fixed $\sigma$ over 6 months). In practice,
volatility is **time-varying** — calm and turbulent periods cluster together. The GARCH(1,1) model captures this:

$$\sigma^2_t = \omega + \alpha \cdot \epsilon^2_{t-1} + \beta \cdot \sigma^2_{t-1}$$

| Parameter | Meaning |
|-----------|---------|
| $\omega$ | Long-run baseline variance |
| $\alpha$ | Sensitivity to yesterday's shock (ARCH term) |
| $\beta$  | Persistence of yesterday's variance (GARCH term) |
| $\alpha + \beta$ | Total persistence — how slowly shocks decay |

The model produces a **1-day-ahead forecast of $\sigma$**, which replaces the fixed $\sigma$ in the VaR formula:

$$\text{VaR}^{GARCH}_{\alpha} = -(\mu - z_{\alpha} \cdot \hat{\sigma}_{t+1})$$

### Which stocks get GARCH — and why

We run the following diagnostic tests independently on each stock's 6-month returns:
- **ADF test** — stationarity (required for GARCH)
- **Ljung-Box on squared returns** — volatility clustering
- **Engle's ARCH test** — time-varying variance
- **Jarque-Bera** — fat tails (determines whether to use Normal or Student-t distribution in GARCH)

If a stock shows **no significant ARCH effects** (Engle $p \geq 0.05$, LB-squared $p \geq 0.05$), its volatility is not time-varying and GARCH adds no value — we use historical $\sigma$ instead.

In [ ]:
def run_diagnostics(series, label):
    r = series.dropna()
    adf_p  = adfuller(r)[1]
    lb_p   = acorr_ljungbox(r,    lags=[10])['lb_pvalue'].iloc[0]
    lbsq_p = acorr_ljungbox(r**2, lags=[10])['lb_pvalue'].iloc[0]
    arch_p = het_arch(r)[1]
    jb_p   = stats.jarque_bera(r)[1]

    results = {
        'Stock'                    : label,
        'Stationarity (ADF)'       : f'{"✓" if adf_p < 0.05 else "✗"} (p={adf_p:.3e})',
        'Vol Clustering (LB²)'     : f'{"✓" if lbsq_p < 0.05 else "✗"} (p={lbsq_p:.3e})',
        'ARCH Effects (Engle)'     : f'{"✓" if arch_p < 0.05 else "✗"} (p={arch_p:.3e})',
        'Fat Tails (JB)'           : f'{"✓" if jb_p < 0.05 else "✗"} (p={jb_p:.3e})',
        'GARCH warranted?'         : 'Yes' if (lbsq_p < 0.05 or arch_p < 0.05) else 'No'
    }
    return results, {
        'arch': arch_p, 'lbsq': lbsq_p,
        'fat_tails': jb_p < 0.05,
        'stationary': adf_p < 0.05
    }

diag_rows = []
diag_flags = {}
for label, col in [('HDFCBANK (Liquid)', 'HDFCBANK'), ('NESTLEIND (Illiquid)', 'NESTLEIND')]:
    row, flags = run_diagnostics(ret[col], label)
    diag_rows.append(row)
    diag_flags[col] = flags

df_diag = pd.DataFrame(diag_rows)
print('Diagnostic Tests — GARCH Applicability')
display(df_diag)

### GARCH Fitting

We fit GARCH(1,1) for each stock where ARCH effects are detected.  
- **Distribution:** Student-t if fat tails detected (JB rejects normality), Normal otherwise  
- **Input:** Log returns scaled to % (standard practice for the `arch` library)  
- **Output:** 1-day-ahead $\sigma$ forecast, converted back to decimal, used in VaR formula

In [ ]:
def fit_garch(series, use_t_dist):
    data   = series.dropna() * 100
    dist   = 't' if use_t_dist else 'normal'
    model  = arch_model(data, vol='Garch', p=1, q=1, dist=dist)
    result = model.fit(disp='off')
    forecast  = result.forecast(horizon=1)
    sigma_fwd = np.sqrt(forecast.variance.values[-1, 0]) / 100
    alpha = result.params['alpha[1]']
    beta  = result.params['beta[1]']
    return result, sigma_fwd, alpha, beta

garch_sigma = {}
garch_rows  = []

for label, col in [('HDFCBANK (Liquid)', 'HDFCBANK'), ('NESTLEIND (Illiquid)', 'NESTLEIND')]:
    flags = diag_flags[col]
    arch_detected = flags['arch'] < 0.05 or flags['lbsq'] < 0.05

    if arch_detected:
        result, sigma_fwd, alpha, beta = fit_garch(ret[col], use_t_dist=flags['fat_tails'])
        garch_sigma[col] = sigma_fwd
        hist_sigma       = ret[col].std()
        mu               = ret[col].mean()
        dist_used        = 'Student-t' if flags['fat_tails'] else 'Normal'

        print(f'{label}  |  Distribution: {dist_used}')
        print(f'  α={alpha:.4f}  β={beta:.4f}  α+β={alpha+beta:.4f}')
        print(f'  GARCH forecast σ : {sigma_fwd*100:.4f}%')
        print(f'  Historical σ     : {hist_sigma*100:.4f}%')
        print()

        for conf, z in [('95%', Z_95), ('99%', Z_99)]:
            var_g = -(mu - z * sigma_fwd)
            var_b, _, _ = parametric_var(ret[col], z)
            garch_rows.append({
                'Stock'          : label,
                'Confidence'     : conf,
                'Dist Used'      : dist_used,
                'Basic VaR (%)'  : round(var_b * 100, 4),
                'GARCH σ (%)'    : round(sigma_fwd * 100, 4),
                'GARCH VaR (%)'  : round(var_g * 100, 4),
                'Difference (%)'  : round((var_g - var_b) * 100, 4)
            })
    else:
        garch_sigma[col] = ret[col].std()
        print(f'{label}  |  No significant ARCH effects → using historical σ for Monte Carlo')
        print()

if garch_rows:
    df_garch = pd.DataFrame(garch_rows)
    print('GARCH Parametric VaR vs Basic Parametric VaR')
    display(df_garch)

In [ ]:
# Conditional volatility plot for stocks where GARCH was fitted
for label, col in [('HDFCBANK (Liquid)', 'HDFCBANK'), ('NESTLEIND (Illiquid)', 'NESTLEIND')]:
    flags = diag_flags[col]
    if not (flags['arch'] < 0.05 or flags['lbsq'] < 0.05):
        continue

    result, _, _, _ = fit_garch(ret[col], use_t_dist=flags['fat_tails'])
    cond_vol  = (result.conditional_volatility / 100) * np.sqrt(252)
    abs_shock = (ret[col].abs()) * np.sqrt(252)

    plt.figure(figsize=(13, 5))
    plt.plot(cond_vol.index, cond_vol, color='#002060', lw=1.8, label='GARCH Conditional Vol')
    plt.scatter(abs_shock.index, abs_shock, color='#C00000', alpha=0.3, s=20, label='Daily Shocks (annualised)')
    plt.title(f'GARCH(1,1) Conditional Volatility — {label}', fontsize=12, fontweight='bold')
    plt.xlabel('Date'); plt.ylabel('Annualised Volatility')
    plt.legend(); plt.grid(ls=':', alpha=0.5)
    plt.tight_layout(); plt.show()

---
## (Optional) Section E: Monte Carlo VaR

### Theory

GARCH parametric VaR (Section D) still assumes a particular return distribution. Monte Carlo VaR
removes this constraint: instead of a formula, we simulate 10,000 possible 1-day returns and
read the VaR directly from the **loss distribution percentile**.

We run two variants for each stock:

| Variant | Shock | Captures fat tails? |
|---------|-------|---------------------|
| Normal MC | $\epsilon \sim N(0,1)$ | No |
| Student-t MC | $\epsilon \sim t(df=5)$, scaled to unit variance | Yes |

The simulated return is: $r_{sim} = \mu + \sigma \cdot \epsilon$

For Student-t, the raw draw is divided by $\sqrt{df/(df-2)}$ to ensure the shock has unit variance,
preserving $\sigma$ as the scale while retaining heavy tails.

**VaR is read directly:** $\text{VaR}_{95\%} = -\text{percentile}(r_{sim},\ 5)$

### Volatility input per stock
- **HDFCBANK:** GARCH forecast $\sigma$ (if ARCH detected), else historical $\sigma$
- **NESTLEIND:** Same logic — $\sigma$ is determined by the diagnostic results above

The gap between Student-t and Normal MC is the **fat-tail premium** — the extra risk the normal
distribution ignores. This gap is wider at 99% than 95%, precisely where tails matter most.

In [ ]:
np.random.seed(42)
N_SIM = 10_000
DOF   = 5

mc_rows = []

for label, col in [('HDFCBANK (Liquid)', 'HDFCBANK'), ('NESTLEIND (Illiquid)', 'NESTLEIND')]:
    mu       = ret[col].mean()
    sigma_in = garch_sigma[col]
    src      = 'GARCH' if (diag_flags[col]['arch'] < 0.05 or diag_flags[col]['lbsq'] < 0.05) else 'Historical'

    print(f'{label}  |  σ source: {src}  μ={mu*100:.4f}%  σ={sigma_in*100:.4f}%')

    sim_norm = mu + sigma_in * np.random.normal(0, 1, N_SIM)
    t_raw    = np.random.standard_t(df=DOF, size=N_SIM)
    sim_t    = mu + sigma_in * (t_raw / np.sqrt(DOF / (DOF - 2)))

    for dist_label, sim in [('Normal', sim_norm), (f'Student-t (df={DOF})', sim_t)]:
        for conf, pct in [('95%', 5), ('99%', 1)]:
            var_mc = -np.percentile(sim, pct)
            mc_rows.append({
                'Stock'         : label,
                'σ Source'      : src,
                'Distribution'  : dist_label,
                'Confidence'    : conf,
                'MC VaR (%)'    : round(var_mc * 100, 4),
                'MC VaR (Rs.1L)': round(var_mc * INVESTMENT, 2)
            })
    print()

df_mc = pd.DataFrame(mc_rows)
print('Monte Carlo VaR Results')
display(df_mc)

In [ ]:
np.random.seed(42)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for idx, (label, col) in enumerate([('HDFCBANK (Liquid)', 'HDFCBANK'),
                                     ('NESTLEIND (Illiquid)', 'NESTLEIND')]):
    ax       = axes[idx]
    mu       = ret[col].mean()
    sigma_in = garch_sigma[col]

    sim_n = mu + sigma_in * np.random.normal(0, 1, N_SIM)
    sim_t = mu + sigma_in * (np.random.standard_t(df=DOF, size=N_SIM) / np.sqrt(DOF / (DOF - 2)))

    ax.hist(sim_n, bins=80, alpha=0.5, color='steelblue',  density=True, label='Normal shocks')
    ax.hist(sim_t, bins=80, alpha=0.5, color='darkorange', density=True, label=f'Student-t (df={DOF})')
    for conf, pct in [('95%', 5), ('99%', 1)]:
        vn = -np.percentile(sim_n, pct)
        vt = -np.percentile(sim_t, pct)
        ax.axvline(-vn, color='blue', lw=1.5, ls=':',  label=f'Normal VaR {conf}={vn*100:.2f}%')
        ax.axvline(-vt, color='red',  lw=1.5, ls='--', label=f't-dist VaR {conf}={vt*100:.2f}%')
    ax.set_title(f'{label}\n10,000 Simulated Returns', fontsize=11)
    ax.set_xlabel('Simulated 1-Day Return'); ax.set_ylabel('Density'); ax.legend(fontsize=7)

plt.suptitle('Monte Carlo: Normal vs Student-t Distributions', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

## Section F: Final Summary — All Methods

In [ ]:
np.random.seed(42)
summary_rows = []

for label, col in [('HDFCBANK (Liquid)', 'HDFCBANK'), ('NESTLEIND (Illiquid)', 'NESTLEIND')]:
    mu       = ret[col].mean()
    sigma_in = garch_sigma[col]
    ret_n, ret_h, _ = classify_regimes(col)

    sim_n = mu + sigma_in * np.random.normal(0, 1, N_SIM)
    sim_t = mu + sigma_in * (np.random.standard_t(df=DOF, size=N_SIM) / np.sqrt(DOF / (DOF - 2)))

    for conf, z, pct in [('95%', Z_95, 5), ('99%', Z_99, 1)]:
        var_b, _, _ = parametric_var(ret[col], z)
        var_n, _, _ = parametric_var(ret_n, z)
        var_h, _, _ = parametric_var(ret_h, z)
        var_mc_n    = -np.percentile(sim_n, pct)
        var_mc_t    = -np.percentile(sim_t, pct)

        row = {
            'Stock'               : label,
            'Confidence'          : conf,
            'Basic Parametric (%)': round(var_b  * 100, 4),
            'Normal Regime (%)'   : round(var_n  * 100, 4),
            'High-Vol Regime (%)' : round(var_h  * 100, 4),
            'MC Normal (%)'       : round(var_mc_n * 100, 4),
            'MC Student-t (%)'    : round(var_mc_t * 100, 4),
        }

        arch_det = diag_flags[col]['arch'] < 0.05 or diag_flags[col]['lbsq'] < 0.05
        if arch_det:
            row['GARCH Parametric (%)'] = round(-(mu - z * sigma_in) * 100, 4)
        else:
            row['GARCH Parametric (%)'] = 'N/A (no ARCH)'

        summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print('VaR Summary: All Methods Comparison')
display(df_summary)

## Interpretation & Discussion

### 1. Liquid vs Illiquid — VaR Comparison

HDFCBANK (Liquid) shows a higher full-period parametric VaR (2.48% at 95%, 3.44% at 99%) than NESTLEIND (1.88% at 95%, 2.67% at 99%). This is counterintuitive at first glance — the "liquid" stock carries more VaR — but the reason is straightforward: HDFCBANK had a significantly higher daily volatility (σ = 1.41%) compared to NESTLEIND (σ = 1.16%) over the sample period. The model-building VaR is entirely driven by return dispersion; illiquidity (captured via Amihud ratio) affects the price impact per unit traded, not the volatility of daily log returns. NESTLEIND's lower VaR reflects a quieter return distribution during this specific 6-month window, not lower inherent risk.

### 2. Normal vs High-Volatility Regime

Regime-based VaR reveals a sharp divergence between calm and stressed periods. For HDFCBANK, 95% VaR rises from 1.42% (Normal) to 4.39% (High-Vol) — a 3× increase. For NESTLEIND, the move is more modest: 1.76% to 2.20% at 95%. This difference in regime sensitivity reflects HDFCBANK's stronger volatility clustering: its 20-day rolling σ fluctuates widely (0.5% to 2.5%+), while NESTLEIND's is more range-bound. The full-period VaR (2.48% for HDFCBANK) sits between these two extremes — it underestimates risk on High-Vol days and overstates it on Normal days. A risk manager relying on the static 6-month figure would be under-reserved by roughly 77% during stressed periods for HDFCBANK.

### 3. Normality Test (Jarque-Bera)

Both stocks reject the null of normality at any conventional significance level. HDFCBANK has a JB statistic of 63.26 (p ≈ 0), driven primarily by excess kurtosis of 3.72 — returns have fatter tails than a normal distribution implies. NESTLEIND has a JB of 28.88 (p ≈ 0), with positive skew of 0.69 in addition to excess kurtosis of 2.09. The practical implication: the 99% VaR from basic parametric VaR is likely understated for both stocks, since the normal distribution underweights extreme losses. This motivates the GARCH and Monte Carlo extensions.

### 4. GARCH(1,1) — HDFCBANK Only

Diagnostic tests confirm ARCH effects in HDFCBANK (Engle p ≈ 0, LB-squared p ≈ 0) but not in NESTLEIND (both p > 0.33). GARCH is therefore fitted only for HDFCBANK, using a Student-t distribution given its fat tails.

The fitted parameters are α = 0.1145, β = 0.8855, giving α + β = 1.000 — at the boundary of integrated GARCH (IGARCH). This near-unit persistence means shocks to variance decay very slowly; elevated volatility from a given day has a prolonged effect on forecasted risk. The GARCH 1-day-ahead σ forecast is 2.74%, nearly double the 6-month historical σ of 1.41%, indicating that HDFCBANK's recent volatility (late March–April 2026, visible in the rolling vol chart) is substantially higher than the 6-month average. As a result, GARCH VaR (4.67% at 95%, 6.54% at 99%) is roughly 88% higher than basic parametric VaR — a material difference for any capital calculation. NESTLEIND's volatility showed no statistically significant clustering, so historical σ is retained for that stock.

### 5. Monte Carlo VaR

Monte Carlo confirms the GARCH parametric estimates for HDFCBANK. The Normal MC (4.70% at 95%, 6.52% at 99%) is nearly identical to GARCH parametric (4.67%, 6.54%) — as expected, since both use the same σ and the same distributional assumption; the small difference is simulation noise from 10,000 draws. The Student-t MC (df = 5) is slightly lower at 95% (4.37%) but higher at 99% (7.18%), which is characteristic of heavy-tailed distributions: they have a fatter centre mass at moderate quantiles and heavier tails at extreme quantiles. The fat-tail premium at 99% for HDFCBANK is approximately 65 bps (7.18% vs 6.52%), representing the additional capital requirement if one accounts for the non-normality of shocks. For NESTLEIND, differences across methods are small (Student-t 99% MC: 3.02% vs Normal 99% MC: 2.72%), consistent with its lower and more stable volatility.

### 6. Stability of VaR Estimates

NESTLEIND's VaR is stable across methods — the spread between lowest (Normal Regime 95%: 1.76%) and highest (Student-t 99% MC: 3.02%) estimates is contained. This stability reflects the absence of ARCH effects and moderate kurtosis. HDFCBANK's estimates are highly sensitive to model choice — basic parametric gives 2.48% at 95% while GARCH-based methods give 4.37–4.70%. This instability is not a modelling flaw; it correctly captures that HDFCBANK's recent volatility regime is elevated relative to its 6-month average. Single-model VaR for this stock would be unreliable for risk management decisions.
